# Weather-Aware Technician Planning Notebook

This notebook is built from the current `field_planner` project. It mixes short explanations with runnable code cells so you can inspect how the system is configured, how weather data is prepared, how risk is predicted, and how the planner generates a final technician schedule.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'docs':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from field_planner.config import DEFAULT_SERVICE_TOWNS, DEFAULT_SETTINGS, SITE_METADATA
from field_planner.data import get_default_tasks, simulate_weather_rows, tasks_to_frame
from field_planner.model import ensure_model, predict_risk
from field_planner.service import plan_day
from field_planner.weather import WeatherAPIClient, get_nakuru_county_forecast


## 1. Planner configuration

The planner is configured for Nakuru County, Kenya. The following cell shows the default service towns, start location, and site metadata used in the current implementation.

In [ ]:
print('Start location:', DEFAULT_SETTINGS.start_location)
print('Work hours:', DEFAULT_SETTINGS.work_hours)
print('Service towns:', DEFAULT_SERVICE_TOWNS)

pd.DataFrame(
    [
        {
            'name': meta.name,
            'query_name': meta.query_name,
            'latitude': meta.latitude,
            'longitude': meta.longitude,
            'exposure_hint': meta.exposure_hint,
            'site_class': meta.site_class,
        }
        for meta in SITE_METADATA.values()
    ]
)


## 2. Synthetic training-style data

The project can generate labeled synthetic weather rows. These are useful for retraining the model or for understanding the feature structure expected by the classifier.

In [ ]:
training_preview = simulate_weather_rows(n_rows=200, seed=42)
print(training_preview.shape)
training_preview.head()


## 3. Default field tasks

Tasks are represented using the `Task` dataclass. Each task has a location, priority, duration, and an `is_outdoor` flag that matters during weather-aware scheduling.

In [ ]:
default_tasks = get_default_tasks()
tasks_to_frame(default_tasks)


## 4. Live weather acquisition

The next cell uses the project weather client. In `auto` mode, the planner will try Open-Meteo first and switch to deterministic fallback data if the API or geocoding is unavailable.

In [ ]:
client = WeatherAPIClient()
forecast_bundle = get_nakuru_county_forecast(forecast_mode='auto', client=client)
print('Forecast source:', forecast_bundle.source)
print('Messages:')
for message in forecast_bundle.messages:
    print('-', message)

forecast_bundle.forecast.head()


## 5. Risk prediction

The system loads the saved model artifact if it exists. If it does not exist and retraining is disabled, the planner falls back to a rule-based risk classifier that still outputs `safe`, `risky`, and `unsafe`.

In [ ]:
model, metadata, retrained = ensure_model(retrain_if_missing=True)
forecast_with_risk = predict_risk(model, forecast_bundle.forecast)
print('Model retrained during load:', retrained)
forecast_with_risk[['time', 'location', 'rain_prob', 'wind_kph', 'gust_kph', 'pred_risk']].head(12)


## 6. End-to-end planning

This final workflow runs the complete system: weather acquisition, feature preparation, risk prediction, baseline scoring, beam-search optimization, and result packaging.

In [ ]:
result = plan_day(forecast_mode='auto', retrain_if_missing=True)
print('Forecast source:', result['forecast_source'])
print('AI score:', round(result['ai_score'], 2))
print('Baseline score:', round(result['baseline_score'], 2))
print('Rule-based fallback used:', result['using_rule_fallback'])

for message in result['messages']:
    print('-', message)


## 7. Recommended schedule and postponed tasks

The planner returns structured tables that can be displayed in Jupyter just as they are shown in the Streamlit app.

In [ ]:
print('Scheduled tasks')
display(result['scheduled_tasks'])

print('Postponed tasks')
display(result['postponed_tasks'])

print('Simulation summary')
display(result['summary'])


## 8. Notes

- If Open-Meteo is reachable, the notebook will use real hourly forecast data.
- If geocoding or forecast retrieval fails, the planner will still run using deterministic fallback weather.
- The same modules used here are the ones used by the Streamlit app, so this notebook is a code-aligned walkthrough of the current project rather than a separate reimplementation.